# ACS Data Preparation

Downloads ACS tract-level indicators, maps them to Chicago community areas, and exports the community-level socioeconomic feature file used by the TDA notebooks.


In [ ]:
# Reproducible path configuration for Colab and local runs
from pathlib import Path
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/tda_crime')
else:
    cwd = Path.cwd()
    PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

def first_existing_path(*candidates):
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists():
            return candidate
    return Path(candidates[0])

BOUNDARIES_PATH = first_existing_path(
    DATA_DIR / 'Boundaries_-_Community_Areas_20260506.geojson',
    PROJECT_ROOT / 'Boundaries_-_Community_Areas_20260506.geojson',
)
ACS_FEATURES_PATH = DATA_DIR / 'ACS_ChicagoFeatures_2019_2023.csv'

print(f'Project root: {PROJECT_ROOT}')
print(f'Data directory: {DATA_DIR}')


## Imports and ACS Variable Definitions


In [ ]:
import pandas as pd
import requests
import geopandas as gpd
import io
import numpy as np
from shapely.geometry import shape

In [ ]:
ACS_VARS = {
    "B01003_001E": "total_population",

    "B17001_001E": "poverty_universe",
    "B17001_002E": "poverty_count",

    "B23025_003E": "labor_force",
    "B23025_005E": "unemployed",

    "B15003_001E": "education_25plus_total",
    "B15003_022E": "bachelors",
    "B15003_023E": "masters",
    "B15003_024E": "professional",
    "B15003_025E": "doctorate",

    "B19013_001E": "median_household_income",

    "B25003_001E": "occupied_housing_units",
    "B25003_002E": "owner_occupied",
    "B25003_003E": "renter_occupied",

    "B25002_001E": "total_housing_units",
    "B25002_003E": "vacant_housing_units",

    "B03002_001E": "race_eth_total",
    "B03002_003E": "white_non_hispanic",
    "B03002_004E": "black_non_hispanic",
    "B03002_006E": "asian_non_hispanic",
    "B03002_012E": "hispanic",
}



In [ ]:
def download_acs_tracts(year, state="17", county="031", vars_dict=ACS_VARS):
    variables = ["NAME"] + list(vars_dict.keys())
    get_vars = ",".join(variables)

    url = (
        f"https://api.census.gov/data/{year}/acs/acs5"
        f"?get={get_vars}"
        f"&for=tract:*"
        f"&in=state:{state}%20county:{county}"
    )

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(data[1:], columns=data[0])

    # Rename ACS variables
    df = df.rename(columns=vars_dict)
    

    # Numeric conversion
    for col in vars_dict.values():
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # GEOID: state + county + tract
    df["tract_geoid"] = df["state"] + df["county"] + df["tract"]
    df["acs_year"] = year

    return df

## Download ACS Tract Data


In [ ]:
acs_2019 = download_acs_tracts(2019)
acs_2021 = download_acs_tracts(2021)
acs_2023 = download_acs_tracts(2023)

acs_tracts = pd.concat([acs_2019, acs_2021, acs_2023], ignore_index=True)
acs_tracts.head()

In [ ]:
def add_acs_rates(df):
    df = df.copy()

    df["poverty_rate"] = df["poverty_count"] / df["poverty_universe"]
    df["unemployment_rate"] = df["unemployed"] / df["labor_force"]

    df["bachelors_plus"] = (df["bachelors"] +df["masters"] +df["professional"] +df["doctorate"])
    df["bachelors_plus_rate"] = df["bachelors_plus"] / df["education_25plus_total"]

    df["renter_rate"] = df["renter_occupied"] / df["occupied_housing_units"]
    df["owner_rate"] = df["owner_occupied"] / df["occupied_housing_units"]

    df["vacancy_rate"] = df["vacant_housing_units"] / df["total_housing_units"]

    df["white_non_hispanic_rate"] = df["white_non_hispanic"] / df["race_eth_total"]
    df["black_non_hispanic_rate"] = df["black_non_hispanic"] / df["race_eth_total"]
    df["asian_non_hispanic_rate"] = df["asian_non_hispanic"] / df["race_eth_total"]
    df["hispanic_rate"] = df["hispanic"] / df["race_eth_total"]

    return df

acs_tracts = add_acs_rates(acs_tracts)

In [ ]:
#Poligons

ca = gpd.read_file(BOUNDARIES_PATH)


## Community Area Boundaries


In [ ]:
print(ca.columns.tolist())
print(ca.head())

In [ ]:

possible_cols = ["area_numbe", "area_num_1", "area_num", "area"]
matches = [c for c in possible_cols if c in ca.columns]

id_col = matches[0]
ca["Community"] = ca[id_col].astype(str).str.extract(r"(\d+)")[0].astype(int)
ca = ca[["Community", "geometry"]].copy()

In [ ]:
def load_cook_tracts_tiger(year):
    url = f"https://www2.census.gov/geo/tiger/TIGER{year}/TRACT/tl_{year}_17_tract.zip"
    tracts = gpd.read_file(url)
    tracts = tracts[tracts["COUNTYFP"] == "031"].copy()  # Cook County
    tracts["tract_geoid"] = tracts["GEOID"].astype(str)
    return tracts[["tract_geoid", "geometry"]]

tracts_2019 = load_cook_tracts_tiger(2019)
tracts_2021 = load_cook_tracts_tiger(2021)
tracts_2023 = load_cook_tracts_tiger(2023)

In [ ]:
def build_tract_to_community_weights(tracts, ca, acs_year):
    # CRS proyectado para área. Illinois East works well for Chicago.
    tracts_p = tracts.to_crs(epsg=3435)
    ca_p = ca.to_crs(epsg=3435)

    tracts_p["tract_area"] = tracts_p.geometry.area

    inter = gpd.overlay(
        tracts_p,
        ca_p,
        how="intersection"
    )

    inter["intersection_area"] = inter.geometry.area

    inter = inter.merge(
        tracts_p[["tract_geoid", "tract_area"]],
        on="tract_geoid",
        how="left",
        suffixes=("", "_tract")
    )

    inter["area_weight"] = inter["intersection_area"] / inter["tract_area"]
    inter["acs_year"] = acs_year

    return inter[["acs_year", "tract_geoid", "Community", "area_weight"]]

w_2019 = build_tract_to_community_weights(tracts_2019, ca, 2019)
w_2021 = build_tract_to_community_weights(tracts_2021, ca, 2021)
w_2023 = build_tract_to_community_weights(tracts_2023, ca, 2023)

tract_ca_weights = pd.concat([w_2019, w_2021, w_2023], ignore_index=True)

## Area-Weighted Tract-to-Community Crosswalk


In [ ]:
acs_weighted = acs_tracts.merge(
    tract_ca_weights,
    on=["acs_year", "tract_geoid"],
    how="inner"
)

In [ ]:
# ACS features at Community Area level
count_cols = [
    "total_population",
    "poverty_universe", "poverty_count",
    "labor_force", "unemployed",
    "education_25plus_total",
    "bachelors", "masters", "professional", "doctorate",
    "occupied_housing_units", "owner_occupied", "renter_occupied",
    "total_housing_units", "vacant_housing_units",
    "race_eth_total",
    "white_non_hispanic", "black_non_hispanic",
    "asian_non_hispanic", "hispanic"
]

# aggregate weighted ACS counts to community area
acs_ca_counts = (
    acs_weighted
    .groupby(["acs_year", "Community"], as_index=False)[count_cols]
    .sum()
)

# create rates safely
acs_ca = acs_ca_counts.copy()

def safe_divide(num, den):
    return np.where(den > 0, num / den, np.nan)

acs_ca["poverty_rate"] = safe_divide(acs_ca["poverty_count"],acs_ca["poverty_universe"])

acs_ca["unemployment_rate"] = safe_divide(acs_ca["unemployed"],acs_ca["labor_force"])

acs_ca["bachelors_plus"] = (acs_ca["bachelors"] +acs_ca["masters"] +acs_ca["professional"] +acs_ca["doctorate"])

acs_ca["bachelors_plus_rate"] = safe_divide(acs_ca["bachelors_plus"],acs_ca["education_25plus_total"])

acs_ca["renter_rate"] = safe_divide(acs_ca["renter_occupied"],acs_ca["occupied_housing_units"])

acs_ca["owner_rate"] = safe_divide(acs_ca["owner_occupied"],acs_ca["occupied_housing_units"])

acs_ca["vacancy_rate"] = safe_divide(acs_ca["vacant_housing_units"],acs_ca["total_housing_units"])

acs_ca["white_non_hispanic_rate"] = safe_divide(acs_ca["white_non_hispanic"],acs_ca["race_eth_total"])

acs_ca["black_non_hispanic_rate"] = safe_divide(acs_ca["black_non_hispanic"],acs_ca["race_eth_total"])

acs_ca["asian_non_hispanic_rate"] = safe_divide(acs_ca["asian_non_hispanic"],acs_ca["race_eth_total"])

acs_ca["hispanic_rate"] = safe_divide(acs_ca["hispanic"],acs_ca["race_eth_total"])

# clean and approximate median household income
# Important: median income (approximation)
acs_income = acs_weighted.copy()

acs_income["median_household_income"] = pd.to_numeric(
    acs_income["median_household_income"],
    errors="coerce"
)

#correct
acs_income.loc[
    acs_income["median_household_income"] <= 0,
    "median_household_income"
] = np.nan

# Use occupied_housing_units as weight.
# At this point occupied_housing_units is already area-weighted,
# so do NOT multiply by area_weight again.
acs_income = acs_income.dropna(subset=["median_household_income"]).copy()

acs_income["income_weight"] = acs_income["occupied_housing_units"]

acs_income["weighted_income"] = (
    acs_income["median_household_income"] *
    acs_income["income_weight"]
)

income_ca = (
    acs_income
    .groupby(["acs_year", "Community"], as_index=False)
    .agg(
        weighted_income_sum=("weighted_income", "sum"),
        income_weight_sum=("income_weight", "sum")
    )
)

income_ca["median_household_income_approx"] = safe_divide(
    income_ca["weighted_income_sum"],
    income_ca["income_weight_sum"]
)

acs_ca = acs_ca.merge(
    income_ca[["acs_year", "Community", "median_household_income_approx"]],
    on=["acs_year", "Community"],
    how="left"
)

# Optional: keep only useful final columns
acs_feature_cols = [
    "acs_year", "Community",

    # optional raw denominators / scale variables
    "total_population",
    "occupied_housing_units",
    "total_housing_units",

    # rates
    "poverty_rate",
    "unemployment_rate",
    "bachelors_plus_rate",
    "renter_rate",
    "owner_rate",
    "vacancy_rate",
    "white_non_hispanic_rate",
    "black_non_hispanic_rate",
    "asian_non_hispanic_rate",
    "hispanic_rate",

    # approximate median
    "median_household_income_approx"
]

acs_ca_final = acs_ca[acs_feature_cols].copy()

acs_ca_final.head()

In [ ]:
acs_ca.columns


In [ ]:
rate_cols = [
    "poverty_rate",
    "unemployment_rate",
    "bachelors_plus_rate",
    "renter_rate",
    "owner_rate",
    "vacancy_rate",
    "white_non_hispanic_rate",
    "black_non_hispanic_rate",
    "asian_non_hispanic_rate",
    "hispanic_rate"
]

validation = pd.DataFrame({
    "min": acs_ca_final[rate_cols + ["median_household_income_approx"]].min(),
    "max": acs_ca_final[rate_cols + ["median_household_income_approx"]].max(),
    "missing": acs_ca_final[rate_cols + ["median_household_income_approx"]].isna().sum()
})

validation

## Export Community-Level ACS Features


In [ ]:
acs_ca_final.to_csv(ACS_FEATURES_PATH, index=False)
